# ArchTune on Colab

The whole project in one notebook. `MINI = True` runs every step on two shapes and twelve configs,
which takes a few minutes and is a cheap way to check the setup. `MINI = False` is the real run.
Steps that have already finished are read back from Drive, so the notebook can be re-run after a
disconnect.

In [ ]:
# Setup: mount Drive, make sure the pinned torch/triton are installed, import archtune.
import importlib.metadata as md, os, subprocess, sys
PROJECT = "/content/drive/MyDrive/archtune"
NEED_GPU_STACK = True

if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

def _ver(pkg):
    try:
        return md.version(pkg).split("+")[0]
    except md.PackageNotFoundError:
        return None

if NEED_GPU_STACK and (_ver("torch"), _ver("triton")) != ("2.6.0", "3.2.0"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "triton==3.2.0"], check=True)
    if "torch" in sys.modules:
        raise SystemExit("torch was already imported: restart the session and run this cell again")

from archtune import config, gpu, model, compiler, run
ENV = gpu.capture()
MINI = config.mini_mode()
RESULTS = os.environ.get("ARCHTUNE_RESULTS") or os.path.join(PROJECT, "results_mini" if MINI else "results")
print(ENV["gpu_name"], "| torch", ENV["torch_version"], "| triton", ENV["triton_version"], "|", RESULTS)
if NEED_GPU_STACK and not gpu.pinned_ok(ENV):
    raise SystemExit("wrong torch/triton loaded: restart the session and run this cell again")

In [ ]:
MINI = True
RUN_PREFLIGHT = MINI
DO = {
    "phase0_smoke": True,
    "phase2_sweep": True,
    "phase3_ir": True,
    "phase4_5_analysis": True,
    "phase5_spotcheck": True,
    "phase6_experiments": True,
    "phase8_integration": True,   # includes the unpruned autotune baseline (slow)
    "phase9_liveness": True,
    "report": True,
}
INDUCTOR_BASELINE = True
R = run.setup(PROJECT, MINI)

In [ ]:
if RUN_PREFLIGHT:
    import subprocess, sys
    proc = subprocess.Popen([sys.executable, "-m", "archtune.run", "preflight", "--out", f"{PROJECT}/results_preflight"],
                            cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    assert proc.wait() == 0, f"preflight failed: {PROJECT}/results_preflight/{ENV['gpu_tag']}/preflight.md"

## Setup checks and timing noise

In [ ]:
if DO["phase0_smoke"]:
    run.timed("setup checks", run.phase0_smoke, R)

## Exhaustive sweep

In [ ]:
if DO["phase2_sweep"]:
    run.timed("sweep", run.phase2_sweep, R)

## Fast vs slow config in IR / PTX / SASS

In [ ]:
if DO["phase3_ir"]:
    display(run.timed("IR", run.phase3_ir, R))

## Predictor accuracy and search strategies

In [ ]:
if DO["phase4_5_analysis"]:
    run.timed("analysis", run.phase4_5_analysis, R)
    from IPython.display import Image
    tag = R.env["gpu_tag"]
    display(Image(str(R.figures / f"fig1_quality_vs_cost_{tag}.png")))
    display(Image(str(R.figures / f"fig2_predictor_{tag}.png")))

In [ ]:
if DO["phase5_spotcheck"]:
    display(run.timed("live check", run.phase5_spotcheck, R))

## Register caps and pipelining depth

In [ ]:
if DO["phase6_experiments"]:
    run.timed("experiments", run.phase6_experiments, R)

## ArchTune inside `triton.autotune`

In [ ]:
if DO["phase8_integration"]:
    display(run.timed("integration", run.phase8_integration, R, inductor=INDUCTOR_BASELINE))

## PTX liveness vs ptxas

In [ ]:
if DO["phase9_liveness"]:
    run.timed("liveness", run.phase9_liveness, R)

## Report

In [ ]:
if DO["report"]:
    print(run.timed("report", run.make_report, R)[:4000])